In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data" / "processed" / "week-3"

NODE_FILE = DATA_DIR / "final_graph_nodes.csv"
EDGE_FILE = DATA_DIR / "final_graph_edges.csv"
TARGET_FILE = DATA_DIR / "final_graph_targets.csv"

EMBEDDING_FILE = DATA_DIR / "spatial_embeddings.csv"

In [3]:
nodes_df = pd.read_csv(NODE_FILE)
edges_df = pd.read_csv(EDGE_FILE)
targets_df = pd.read_csv(TARGET_FILE)
embeddings_df = pd.read_csv(EMBEDDING_FILE)

print("Final nodes:", nodes_df.shape)
print("Final edges:", edges_df.shape)
print("Final targets:", targets_df.shape)
print("Spatial embeddings:", embeddings_df.shape)

Final nodes: (21613, 26)
Final edges: (108065, 3)
Final targets: (21613, 2)
Spatial embeddings: (21613, 9)


In [4]:
assert len(nodes_df) == 21613
assert len(targets_df) == len(nodes_df)
assert len(embeddings_df) == len(nodes_df)

print("✅ Node and embedding counts validated.")

✅ Node and embedding counts validated.


In [5]:
assert nodes_df["node_id"].nunique() == len(nodes_df)

assert embeddings_df["node_id"].nunique() == len(embeddings_df)

assert targets_df["node_id"].nunique() == len(targets_df)

print("✅ No duplicate node IDs detected.")

✅ No duplicate node IDs detected.


In [6]:
node_ids = set(nodes_df["node_id"])
embedding_ids = set(embeddings_df["node_id"])
target_ids = set(targets_df["node_id"])

assert node_ids == embedding_ids
assert node_ids == target_ids

print("✅ Node, embedding and target IDs are fully aligned.")

✅ Node, embedding and target IDs are fully aligned.


In [7]:
valid_ids = set(nodes_df["node_id"])

assert edges_df["source"].isin(valid_ids).all()
assert edges_df["target"].isin(valid_ids).all()

print("✅ All graph edges reference valid nodes.")

✅ All graph edges reference valid nodes.


In [8]:
self_loops = (edges_df["source"] == edges_df["target"]).sum()

print("Self-loops:", self_loops)

assert self_loops == 0

print("✅ Self-loop validation passed.")

Self-loops: 0
✅ Self-loop validation passed.


In [9]:
K = 5

expected_edges = len(nodes_df) * K

print("Expected edges:", expected_edges)
print("Actual edges:", len(edges_df))

assert len(edges_df) == expected_edges

print("✅ Total KNN edge count validated.")

Expected edges: 108065
Actual edges: 108065
✅ Total KNN edge count validated.


In [10]:
outgoing_degree = (edges_df.groupby("source").size())

print(outgoing_degree.describe())

assert len(outgoing_degree) == len(nodes_df)
assert (outgoing_degree == K).all()

print("✅ Every node has exactly 5 outgoing neighbors.")

count    21613.0
mean         5.0
std          0.0
min          5.0
25%          5.0
50%          5.0
75%          5.0
max          5.0
dtype: float64
✅ Every node has exactly 5 outgoing neighbors.


In [11]:
assert "distance_km" in edges_df.columns

assert edges_df["distance_km"].notna().all()
assert np.isfinite(edges_df["distance_km"]).all()
assert (edges_df["distance_km"] >= 0).all()

print("✅ Edge distance values validated.")

✅ Edge distance values validated.


In [12]:
feature_columns = [col for col in nodes_df.columns if col != "node_id"]

print("Number of node features:", len(feature_columns))

print("\nMissing feature values:")
print(nodes_df[feature_columns].isnull().sum().sum())

assert (nodes_df[feature_columns].isnull().sum().sum() == 0)

print("✅ Node feature validation passed.")

Number of node features: 25

Missing feature values:
0
✅ Node feature validation passed.


In [13]:
embedding_columns = [col for col in embeddings_df.columns if col != "node_id"]

print("Embedding dimensions:", len(embedding_columns))

assert len(embedding_columns) == 8

assert (embeddings_df[embedding_columns].isnull().sum().sum() == 0)

assert np.isfinite(embeddings_df[embedding_columns].to_numpy()).all()

print("✅ Spatial embedding validation passed.")

Embedding dimensions: 8
✅ Spatial embedding validation passed.


In [14]:
print("Target columns:")
print(targets_df.columns.tolist())

print("\nMissing target values:")
print(targets_df.isnull().sum())

assert targets_df.isnull().sum().sum() == 0

print("✅ Target validation passed.")

Target columns:
['node_id', 'price']

Missing target values:
node_id    0
price      0
dtype: int64
✅ Target validation passed.


In [15]:
TARGET_COLUMN = "price"

assert TARGET_COLUMN in targets_df.columns

assert TARGET_COLUMN not in feature_columns

print("✅ Target leakage check passed.")

✅ Target leakage check passed.


In [16]:
for file_path in [NODE_FILE,EDGE_FILE,TARGET_FILE,EMBEDDING_FILE]:
    assert file_path.exists()
    print(f"✅ Verified: {file_path}")

✅ Verified: ..\data\processed\week-3\final_graph_nodes.csv
✅ Verified: ..\data\processed\week-3\final_graph_edges.csv
✅ Verified: ..\data\processed\week-3\final_graph_targets.csv
✅ Verified: ..\data\processed\week-3\spatial_embeddings.csv


In [17]:
print("=" * 60)
print("WEEK 3 FINAL DATASET SUMMARY")
print("=" * 60)

print(f"Nodes: {len(nodes_df):,}")
print(f"Edges: {len(edges_df):,}")
print(f"K: {K}")
print(f"Node features: {len(feature_columns)}")
print(f"Spatial embedding dimensions: {len(embedding_columns)}")
print(f"Targets: {len(targets_df):,}")
print(f"Self-loops: {self_loops}")

print("\nValidation:")
print("✓ Node IDs aligned")
print("✓ Embedding IDs aligned")
print("✓ Target IDs aligned")
print("✓ Graph edges valid")
print("✓ No self-loops")
print("✓ KNN edge count valid")
print("✓ KNN outgoing degree valid")
print("✓ Edge distances valid")
print("✓ Node features valid")
print("✓ Spatial embeddings valid")
print("✓ Targets valid")
print("✓ Target leakage check passed")
print("✓ Required files exist")

print("\n🎉 WEEK 3 VALIDATION PASSED")

WEEK 3 FINAL DATASET SUMMARY
Nodes: 21,613
Edges: 108,065
K: 5
Node features: 25
Spatial embedding dimensions: 8
Targets: 21,613
Self-loops: 0

Validation:
✓ Node IDs aligned
✓ Embedding IDs aligned
✓ Target IDs aligned
✓ Graph edges valid
✓ No self-loops
✓ KNN edge count valid
✓ KNN outgoing degree valid
✓ Edge distances valid
✓ Node features valid
✓ Spatial embeddings valid
✓ Targets valid
✓ Target leakage check passed
✓ Required files exist

🎉 WEEK 3 VALIDATION PASSED
